<a href="https://colab.research.google.com/github/Aman6665/NLP-Program-using-Python-NLTK-and-spaCy/blob/main/23_Implementing_Attention_Mechanism_(Basic_custom_model).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Implementing a Custom Attention Mechanism (Bahdanau/Additive Attention)

This notebook demonstrates a custom implementation of the Bahdanau Attention mechanism in PyTorch.

We define a custom `Attention` class that:
1. Takes the encoder outputs and decoder hidden states.
2. Computes the alignment scores using a feed-forward network layer.
3. Applies a Softmax function to obtain the attention weights.
4. Computes the context vector as a weighted sum of the encoder outputs.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CustomAttention(nn.Module):
    def __init__(self, encoder_hidden_dim, decoder_hidden_dim, attention_dim):
        super(CustomAttention, self).__init__()
        # Linear layers to project encoder and decoder hidden states
        self.W_encoder = nn.Linear(encoder_hidden_dim, attention_dim, bias=False)
        self.W_decoder = nn.Linear(decoder_hidden_dim, attention_dim, bias=False)
        # Linear layer to produce the final attention score
        self.V = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        """
        Args:
            decoder_hidden: Tensor of shape [batch_size, decoder_hidden_dim] (current query state)
            encoder_outputs: Tensor of shape [batch_size, seq_len, encoder_hidden_dim] (keys and values)
        Returns:
            context_vector: Tensor of shape [batch_size, encoder_hidden_dim]
            attention_weights: Tensor of shape [batch_size, seq_len, 1]
        """
        # Unsqueeze decoder hidden state to make it [batch_size, 1, decoder_hidden_dim]
        decoder_hidden_expanded = decoder_hidden.unsqueeze(1)

        # Project states to the common attention dimension
        # encoder_score shape: [batch_size, seq_len, attention_dim]
        # decoder_score shape: [batch_size, 1, attention_dim]
        encoder_score = self.W_encoder(encoder_outputs)
        decoder_score = self.W_decoder(decoder_hidden_expanded)

        # Bahdanau Additive formulation: score = V^T * tanh(W_encoder * h_enc + W_decoder * s_dec)
        # alignment_scores shape: [batch_size, seq_len, 1]
        alignment_scores = self.V(torch.tanh(encoder_score + decoder_score))

        # Apply softmax over the sequence length dimension (dim=1) to get weights
        attention_weights = F.softmax(alignment_scores, dim=1)

        # Compute context vector as the weighted sum of encoder outputs
        # attention_weights: [batch_size, seq_len, 1]
        # encoder_outputs: [batch_size, seq_len, encoder_hidden_dim]
        context_vector = torch.sum(attention_weights * encoder_outputs, dim=1)

        return context_vector, attention_weights

# --- Demo and Verification ---
if __name__ == "__main__":
    # Define dimensions
    batch_size = 2
    seq_len = 5
    encoder_hidden_dim = 16
    decoder_hidden_dim = 16
    attention_dim = 8

    # Generate dummy tensors
    torch.manual_seed(42)
    toy_encoder_outputs = torch.randn(batch_size, seq_len, encoder_hidden_dim)
    toy_decoder_hidden = torch.randn(batch_size, decoder_hidden_dim)

    # Initialize custom attention layer
    attention_layer = CustomAttention(encoder_hidden_dim, decoder_hidden_dim, attention_dim)

    # Forward pass
    context, weights = attention_layer(toy_decoder_hidden, toy_encoder_outputs)

    print("--- Output Shapes ---")
    print(f"Encoder Outputs: {toy_encoder_outputs.shape}")
    print(f"Decoder Hidden:  {toy_decoder_hidden.shape}")
    print(f"Context Vector:  {context.shape}")
    print(f"Attention Weights:\n{weights.shape}\n")

    print("--- Sample Attention Weights (Batch 0) ---")
    for i, w in enumerate(weights[0]):
        print(f"Token {i}: {w.item():.4f}")
    print(f"Sum of weights (must equal 1): {weights[0].sum().item():.4f}")

--- Output Shapes ---
Encoder Outputs: torch.Size([2, 5, 16])
Decoder Hidden:  torch.Size([2, 16])
Context Vector:  torch.Size([2, 16])
Attention Weights:
torch.Size([2, 5, 1])

--- Sample Attention Weights (Batch 0) ---
Token 0: 0.1826
Token 1: 0.1954
Token 2: 0.1649
Token 3: 0.2849
Token 4: 0.1722
Sum of weights (must equal 1): 1.0000
